# Data Contracts: Formal Agreements Between Data Producers and Consumers

A **data contract** is a formal, machine-enforceable specification that a data pipeline stage promises to deliver: column names, types, value ranges, statistical properties, and business rules. Data contracts make implicit assumptions explicit and catch violations early, before they corrupt downstream ML models or production systems.

This notebook covers schema contracts with Pandera, statistical contracts, row-level contracts with Pydantic, CI/CD integration, contract versioning, and a full pipeline example.

## Why Data Contracts Matter

ML pipelines break **silently** when upstream data changes:

- A data engineer renames a column: `salary` becomes `annual_salary`. Your feature engineering code breaks at runtime.
- A source system starts sending nulls in a previously non-null column. Your model trains on NaN-filled features and performance degrades without any error.
- A categorical column adds a new category. Your one-hot encoder throws an error in production.

**Without contracts**: the violation is discovered at Stage 4 (prediction), after training has already run on corrupted features. The debug loop is days long.

**With contracts**: the violation is caught at Stage 1 (ingestion), before it propagates. The fix takes minutes.

This is the core value of data contracts: **shift the detection of data problems from production to the earliest possible point in the pipeline.**

In [1]:
import pandas as pd
import numpy as np
import datetime
import json
from typing import Optional

print("Core imports loaded")

Core imports loaded


## Section 1: Pandera for Schema Validation

Pandera is a Python library for validating pandas DataFrames. You declare a schema -- column names, types, value constraints -- and Pandera checks that a DataFrame conforms to it. When validation fails, Pandera raises a `SchemaError` with a clear message pointing to exactly which rows and columns violated which check.

In [2]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# Define a data contract for an employee dataset
employee_schema = DataFrameSchema(
    columns={
        "employee_id": Column(
            dtype=str,
            nullable=False,
            checks=[
                Check.str_matches(r"^EMP-\d{5}$"),  # Must match EMP-12345 format
            ],
            description="Unique employee identifier in format EMP-NNNNN",
        ),
        "age": Column(
            dtype=int,
            nullable=False,
            checks=[
                Check.in_range(18, 80),  # Age must be between 18 and 80
            ],
            description="Employee age in years",
        ),
        "salary": Column(
            dtype=float,
            nullable=False,
            checks=[
                Check.in_range(20_000, 500_000),  # Salary range check
            ],
            description="Annual salary in USD",
        ),
        "department": Column(
            dtype=str,
            nullable=False,
            checks=[
                Check.isin(["Engineering", "Sales", "HR", "Finance", "Marketing"]),
            ],
            description="Department name (controlled vocabulary)",
        ),
        "email": Column(
            dtype=str,
            nullable=True,  # Email is optional
            checks=[
                Check.str_matches(r"^[^@]+@[^@]+\.[^@]+$"),  # Basic email format
            ],
            description="Corporate email address",
        ),
    },
    name="employee_schema_v1",
    coerce=False,   # Do NOT silently cast types -- fail explicitly if wrong type
    strict=False,   # Allow extra columns beyond what is declared
)

print("Employee schema defined with checks:")
for col_name, col in employee_schema.columns.items():
    print(f"  {col_name}: {col.dtype}, nullable={col.nullable}, checks={len(col.checks)}")

Employee schema defined with checks:
  employee_id: str, nullable=False, checks=1
  age: int64, nullable=False, checks=1
  salary: float64, nullable=False, checks=1
  department: str, nullable=False, checks=1
  email: str, nullable=True, checks=1


/home/dell/Desktop/AI_Tasks/Additional_Data/zero-to-ai-engineer/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [3]:
# Test with valid data
valid_employees = pd.DataFrame({
    "employee_id": ["EMP-00001", "EMP-00002", "EMP-00003"],
    "age":         [28, 45, 33],
    "salary":      [85000.0, 120000.0, 95000.0],
    "department":  ["Engineering", "Finance", "Sales"],
    "email":       ["alice@company.com", "bob@company.com", None],
})

try:
    validated = employee_schema.validate(valid_employees)
    print("Valid data passed schema validation.")
    print(validated)
except pa.errors.SchemaError as e:
    print(f"Validation failed: {e}")

Valid data passed schema validation.
  employee_id  age    salary   department              email
0   EMP-00001   28   85000.0  Engineering  alice@company.com
1   EMP-00002   45  120000.0      Finance    bob@company.com
2   EMP-00003   33   95000.0        Sales               None


In [4]:
# Test with invalid data -- multiple violations
invalid_employees = pd.DataFrame({
    "employee_id": ["EMP-00001", "BADID",    "EMP-00003"],  # row 1: bad ID format
    "age":         [28,           45,           150],          # row 2: age > 80
    "salary":      [85000.0,      -5000.0,      95000.0],     # row 1: negative salary
    "department":  ["Engineering", "Catering",  "Sales"],    # row 1: unknown department
    "email":       ["alice@company.com", "bob@company.com", None],
})

try:
    employee_schema.validate(invalid_employees, lazy=True)  # lazy=True: collect all errors
    print("Passed (unexpected)")
except pa.errors.SchemaErrors as e:
    print("Schema validation failed with the following errors:")
    print(e.failure_cases)
except pa.errors.SchemaError as e:
    print(f"Schema error: {e}")

Schema validation failed with the following errors:
  schema_context       column  \
0         Column  employee_id   
1         Column          age   
2         Column       salary   
3         Column   department   

                                               check  check_number  \
0                         str_matches('^EMP-\d{5}$')             0   
1                                   in_range(18, 80)             0   
2                            in_range(20000, 500000)             0   
3  isin(['Engineering', 'Sales', 'HR', 'Finance',...             0   

  failure_case  index  
0        BADID      1  
1          150      2  
2      -5000.0      1  
3     Catering      1  


In [5]:
# Using decorators to enforce contracts at function boundaries
# @pa.check_output on a data-producing function
# @pa.check_input on a data-consuming function

@pa.check_output(employee_schema)
def load_employees_from_db() -> pd.DataFrame:
    """Data producer: load employees. Contract enforced on output."""
    # In production, this runs a SQL query.
    # The decorator validates the return value against employee_schema.
    return pd.DataFrame({
        "employee_id": ["EMP-00010", "EMP-00011"],
        "age":         [30, 40],
        "salary":      [70000.0, 90000.0],
        "department":  ["Engineering", "HR"],
        "email":       ["charlie@co.com", "diana@co.com"],
    })

@pa.check_input(employee_schema, obj_getter=0)  # validate first argument
def compute_avg_salary(employees: pd.DataFrame, department: str) -> float:
    """Data consumer: compute average salary. Contract enforced on input."""
    subset = employees[employees["department"] == department]
    return subset["salary"].mean() if len(subset) > 0 else 0.0

# This should work: producer returns valid data, consumer receives valid input
employees = load_employees_from_db()
print(f"Loaded {len(employees)} employees (contract passed).")

avg = compute_avg_salary(employees, "Engineering")
print(f"Average Engineering salary: ${avg:,.0f}")

Loaded 2 employees (contract passed).
Average Engineering salary: $70,000


## Section 2: Statistical Contracts (Distribution Checks)

Schema contracts catch structural violations (wrong type, null where not allowed, value out of range). Statistical contracts catch **distribution violations**: the data has the right shape but the wrong distribution.

Distribution violations are particularly dangerous for ML: a model trained on salary data with mean $80k will behave poorly if production data has mean $500k, even if all individual values are technically valid.

In [6]:
from scipy import stats

def statistical_contract_check(df: pd.DataFrame, reference_df: pd.DataFrame = None) -> dict:
    """Check statistical properties of the employee dataset.
    Returns a dict with check name -> {passed, value, threshold}.
    """
    results = {}

    # Check 1: Mean salary in expected range
    mean_salary = df["salary"].mean()
    results["mean_salary_range"] = {
        "passed": 30_000 <= mean_salary <= 200_000,
        "value": round(mean_salary, 2),
        "threshold": "30000 <= mean <= 200000",
    }

    # Check 2: Missing rate below 5% for salary
    missing_rate = df["salary"].isnull().mean()
    results["salary_missing_rate"] = {
        "passed": missing_rate < 0.05,
        "value": round(missing_rate, 4),
        "threshold": "< 0.05",
    }

    # Check 3: Department distribution has not shifted (chi-square test)
    # Compare observed distribution to expected (from reference data or business rule)
    expected_fractions = {
        "Engineering": 0.40,
        "Sales":       0.25,
        "HR":          0.10,
        "Finance":     0.15,
        "Marketing":   0.10,
    }
    observed_counts = df["department"].value_counts()
    n = len(df)
    expected_counts = [expected_fractions.get(d, 0) * n
                       for d in expected_fractions.keys()]
    observed_aligned = [observed_counts.get(d, 0)
                        for d in expected_fractions.keys()]

    if n >= 5 and sum(expected_counts) > 0:
        chi2_stat, p_value = stats.chisquare(observed_aligned, f_exp=expected_counts)
        dist_ok = p_value > 0.05  # not significantly different
    else:
        p_value = None
        dist_ok = True  # not enough data to test

    results["department_distribution"] = {
        "passed": dist_ok,
        "value": f"p={p_value:.3f}" if p_value is not None else "N/A",
        "threshold": "chi-square p > 0.05",
    }

    return results

# Generate a test dataset
np.random.seed(42)
n = 200
dept_choices = np.random.choice(
    ["Engineering", "Sales", "HR", "Finance", "Marketing"],
    p=[0.40, 0.25, 0.10, 0.15, 0.10],
    size=n,
)
test_df = pd.DataFrame({
    "employee_id": [f"EMP-{i:05d}" for i in range(n)],
    "age":         np.random.randint(22, 65, n),
    "salary":      np.random.uniform(45_000, 180_000, n),
    "department":  dept_choices,
    "email":       [f"emp{i}@co.com" for i in range(n)],
})

print("Statistical contract checks:")
stat_results = statistical_contract_check(test_df)
for check_name, result in stat_results.items():
    status = "PASS" if result["passed"] else "FAIL"
    print(f"  [{status}] {check_name}: value={result['value']}, threshold={result['threshold']}")

Statistical contract checks:
  [PASS] mean_salary_range: value=118086.49, threshold=30000 <= mean <= 200000
  [PASS] salary_missing_rate: value=0.0, threshold=< 0.05
  [PASS] department_distribution: value=p=0.493, threshold=chi-square p > 0.05


## Section 3: Pydantic for Row-Level Contracts

Pandera validates DataFrames (batches). Pydantic validates individual records as they arrive, which is useful for:

- Streaming data pipelines (validate each event before processing)
- API input validation (validate each request payload)
- Real-time ML feature pipelines (validate each sensor reading before feature extraction)

In [7]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ValidationError
from typing import Literal

class SensorReading(BaseModel):
    """Row-level contract for IoT sensor data.
    Every field has a type, and field validators enforce business rules.
    """
    sensor_id: str = Field(..., description="Sensor identifier")
    temperature: float = Field(..., description="Temperature in Celsius")
    humidity: float = Field(..., description="Relative humidity 0-100")
    pressure: float = Field(..., description="Atmospheric pressure in hPa")
    status: Literal["active", "maintenance", "error"] = Field(default="active")
    timestamp: datetime.datetime = Field(..., description="Reading timestamp (UTC)")

    @field_validator("sensor_id")
    @classmethod
    def sensor_id_format(cls, v: str) -> str:
        """Sensor IDs must match format SNS-NNNN."""
        import re
        if not re.match(r"^SNS-\d{4}$", v):
            raise ValueError(f"sensor_id must match SNS-NNNN format, got: {v}")
        return v

    @field_validator("temperature")
    @classmethod
    def temperature_range(cls, v: float) -> float:
        """Temperature must be within physically plausible range for indoor sensors."""
        if not (-40 <= v <= 85):
            raise ValueError(f"Temperature {v}C is outside sensor operating range [-40, 85]")
        return v

    @field_validator("humidity")
    @classmethod
    def humidity_range(cls, v: float) -> float:
        if not (0 <= v <= 100):
            raise ValueError(f"Humidity {v}% is outside valid range [0, 100]")
        return v

    @model_validator(mode="after")
    def check_timestamp_not_future(self) -> "SensorReading":
        """Readings from the future are likely a clock error."""
        now = datetime.datetime.utcnow().replace(tzinfo=datetime.timezone.utc)
        ts = self.timestamp
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=datetime.timezone.utc)
        if ts > now + datetime.timedelta(minutes=5):
            raise ValueError(f"Timestamp {self.timestamp} is in the future")
        return self

print("SensorReading model defined")

# Valid record
valid_record = SensorReading(
    sensor_id="SNS-0042",
    temperature=22.5,
    humidity=55.0,
    pressure=1013.25,
    timestamp=datetime.datetime.utcnow(),
)
print(f"Valid record: {valid_record.sensor_id}, temp={valid_record.temperature}C")

SensorReading model defined
Valid record: SNS-0042, temp=22.5C


/tmp/ipykernel_18738/2163120407.py:59: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp=datetime.datetime.utcnow(),
/tmp/ipykernel_18738/2163120407.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow().replace(tzinfo=datetime.timezone.utc)


In [8]:
# Demonstrate validation errors
invalid_cases = [
    {
        "description": "Bad sensor ID format",
        "data": {
            "sensor_id": "SENSOR_42",  # wrong format
            "temperature": 22.5,
            "humidity": 55.0,
            "pressure": 1013.25,
            "timestamp": datetime.datetime.utcnow(),
        },
    },
    {
        "description": "Temperature out of range",
        "data": {
            "sensor_id": "SNS-0001",
            "temperature": 150.0,  # physically impossible for this sensor
            "humidity": 55.0,
            "pressure": 1013.25,
            "timestamp": datetime.datetime.utcnow(),
        },
    },
]

for case in invalid_cases:
    print(f"\nTesting: {case['description']}")
    try:
        SensorReading(**case["data"])
        print("  UNEXPECTED PASS")
    except ValidationError as e:
        for error in e.errors():
            print(f"  [ValidationError] field={error['loc']}, msg={error['msg']}")


Testing: Bad sensor ID format
  [ValidationError] field=('sensor_id',), msg=Value error, sensor_id must match SNS-NNNN format, got: SENSOR_42

Testing: Temperature out of range
  [ValidationError] field=('temperature',), msg=Value error, Temperature 150.0C is outside sensor operating range [-40, 85]


/tmp/ipykernel_18738/3343943519.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow(),
/tmp/ipykernel_18738/3343943519.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow(),


In [9]:
# Streaming validation: validate each record before processing

def process_sensor_stream(raw_records: list) -> dict:
    """Validate and process a stream of sensor readings.
    Invalid records are quarantined, not silently dropped or propagated.
    """
    valid = []
    quarantined = []

    for raw in raw_records:
        try:
            reading = SensorReading(**raw)
            valid.append(reading)
        except ValidationError as e:
            quarantined.append({
                "raw": raw,
                "errors": [{"field": str(err["loc"]), "msg": err["msg"]}
                           for err in e.errors()],
            })

    # Process only valid records
    avg_temp = sum(r.temperature for r in valid) / len(valid) if valid else None

    return {
        "total": len(raw_records),
        "valid": len(valid),
        "quarantined": len(quarantined),
        "quarantine_rate": len(quarantined) / len(raw_records) if raw_records else 0,
        "avg_temperature": round(avg_temp, 2) if avg_temp is not None else None,
        "quarantined_records": quarantined,
    }

# Simulate a stream with some invalid records
now = datetime.datetime.utcnow()
stream = [
    {"sensor_id": "SNS-0001", "temperature": 21.0, "humidity": 50.0, "pressure": 1012.0, "timestamp": now},
    {"sensor_id": "SNS-0002", "temperature": 23.5, "humidity": 60.0, "pressure": 1011.0, "timestamp": now},
    {"sensor_id": "INVALID",  "temperature": 22.0, "humidity": 55.0, "pressure": 1010.0, "timestamp": now},
    {"sensor_id": "SNS-0003", "temperature": 999.0, "humidity": 45.0, "pressure": 1013.0, "timestamp": now},
    {"sensor_id": "SNS-0004", "temperature": 20.0, "humidity": 52.0, "pressure": 1014.0, "timestamp": now},
]

result = process_sensor_stream(stream)
print("Stream processing result:")
print(f"  Total records:    {result['total']}")
print(f"  Valid:            {result['valid']}")
print(f"  Quarantined:      {result['quarantined']} ({result['quarantine_rate']:.0%})")
print(f"  Avg temperature:  {result['avg_temperature']}C")
print("\nQuarantined records:")
for q in result["quarantined_records"]:
    print(f"  sensor_id={q['raw']['sensor_id']}: {q['errors']}")

Stream processing result:
  Total records:    5
  Valid:            3
  Quarantined:      2 (40%)
  Avg temperature:  21.5C

Quarantined records:
  sensor_id=INVALID: [{'field': "('sensor_id',)", 'msg': 'Value error, sensor_id must match SNS-NNNN format, got: INVALID'}]
  sensor_id=SNS-0003: [{'field': "('temperature',)", 'msg': 'Value error, Temperature 999.0C is outside sensor operating range [-40, 85]'}]


/tmp/ipykernel_18738/1892754213.py:34: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow()
/tmp/ipykernel_18738/2163120407.py:43: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.datetime.utcnow().replace(tzinfo=datetime.timezone.utc)


## Section 4: Contract Testing in CI/CD

Data contracts are most valuable when they run automatically on every code change. By including contract tests in your CI pipeline, you catch data issues before they reach production:

1. Developer commits a change to the feature engineering function
2. CI runs `pytest tests/test_data_contracts.py`
3. The test loads sample data and validates it against the contract
4. If the contract fails, the build fails and the PR cannot be merged

In [10]:
# Example: data contract tests using pytest
# In a real project, these go in tests/test_data_contracts.py

contract_test_example = '''
# tests/test_data_contracts.py
import pytest
import pandas as pd
import pandera as pa
from your_project.schemas import employee_schema
from your_project.pipeline import load_employees_from_db, engineer_features

# conftest.py fixture: sample data that represents typical production data
@pytest.fixture
def sample_employees():
    return pd.DataFrame({
        "employee_id": ["EMP-00001", "EMP-00002", "EMP-00003"],
        "age":         [28, 45, 33],
        "salary":      [85000.0, 120000.0, 95000.0],
        "department":  ["Engineering", "Finance", "Sales"],
        "email":       ["a@co.com", "b@co.com", None],
    })

def test_employee_schema_is_valid(sample_employees):
    """Data producer output must conform to the employee schema contract."""
    # This will raise pa.errors.SchemaError if the contract is violated
    employee_schema.validate(sample_employees)

def test_feature_engineering_preserves_schema(sample_employees):
    """Feature engineering must not break the downstream schema."""
    features = engineer_features(sample_employees)
    assert "salary_normalized" in features.columns
    assert features["salary_normalized"].between(0, 1).all()

def test_schema_rejects_unknown_departments(sample_employees):
    """Schema must reject records with unknown department values."""
    bad_data = sample_employees.copy()
    bad_data.loc[0, "department"] = "Catering"  # not in controlled vocabulary
    with pytest.raises(pa.errors.SchemaError):
        employee_schema.validate(bad_data)

def test_schema_rejects_negative_salary(sample_employees):
    """Schema must reject negative salaries."""
    bad_data = sample_employees.copy()
    bad_data.loc[0, "salary"] = -1000.0
    with pytest.raises(pa.errors.SchemaError):
        employee_schema.validate(bad_data)
'''

print("Example CI/CD contract test file (tests/test_data_contracts.py):")
print(contract_test_example)

Example CI/CD contract test file (tests/test_data_contracts.py):

# tests/test_data_contracts.py
import pytest
import pandas as pd
import pandera as pa
from your_project.schemas import employee_schema
from your_project.pipeline import load_employees_from_db, engineer_features

# conftest.py fixture: sample data that represents typical production data
@pytest.fixture
def sample_employees():
    return pd.DataFrame({
        "employee_id": ["EMP-00001", "EMP-00002", "EMP-00003"],
        "age":         [28, 45, 33],
        "salary":      [85000.0, 120000.0, 95000.0],
        "department":  ["Engineering", "Finance", "Sales"],
        "email":       ["a@co.com", "b@co.com", None],
    })

def test_employee_schema_is_valid(sample_employees):
    """Data producer output must conform to the employee schema contract."""
    # This will raise pa.errors.SchemaError if the contract is violated
    employee_schema.validate(sample_employees)

def test_feature_engineering_preserves_schema(sample_e

In [11]:
# Run contract tests inline to demonstrate the pattern
import traceback

sample_employees = pd.DataFrame({
    "employee_id": ["EMP-00001", "EMP-00002", "EMP-00003"],
    "age":         [28, 45, 33],
    "salary":      [85000.0, 120000.0, 95000.0],
    "department":  ["Engineering", "Finance", "Sales"],
    "email":       ["a@co.com", "b@co.com", None],
})

def run_contract_tests():
    tests_run = 0
    tests_passed = 0

    def test(name, fn):
        nonlocal tests_run, tests_passed
        tests_run += 1
        try:
            fn()
            tests_passed += 1
            print(f"  [PASS] {name}")
        except Exception as e:
            print(f"  [FAIL] {name}: {e}")

    def t_valid_data():
        employee_schema.validate(sample_employees)

    def t_rejects_bad_department():
        bad = sample_employees.copy()
        bad.loc[0, "department"] = "Catering"
        try:
            employee_schema.validate(bad)
            raise AssertionError("Should have raised SchemaError")
        except pa.errors.SchemaError:
            pass  # Expected

    def t_rejects_negative_salary():
        bad = sample_employees.copy()
        bad.loc[0, "salary"] = -1000.0
        try:
            employee_schema.validate(bad)
            raise AssertionError("Should have raised SchemaError")
        except pa.errors.SchemaError:
            pass  # Expected

    def t_rejects_out_of_range_age():
        bad = sample_employees.copy()
        bad.loc[0, "age"] = 150
        try:
            employee_schema.validate(bad)
            raise AssertionError("Should have raised SchemaError")
        except pa.errors.SchemaError:
            pass  # Expected

    test("Valid data passes schema", t_valid_data)
    test("Schema rejects unknown departments", t_rejects_bad_department)
    test("Schema rejects negative salary", t_rejects_negative_salary)
    test("Schema rejects age > 80", t_rejects_out_of_range_age)

    print(f"\nResults: {tests_passed}/{tests_run} passed")
    return tests_passed == tests_run

print("Running data contract tests:")
all_passed = run_contract_tests()
print(f"Build status: {'SUCCESS' if all_passed else 'FAILED'}")

Running data contract tests:
  [PASS] Valid data passes schema
  [PASS] Schema rejects unknown departments
  [PASS] Schema rejects negative salary
  [PASS] Schema rejects age > 80

Results: 4/4 passed
Build status: SUCCESS


## Section 5: Contract Versioning and Evolution

Data contracts change as business requirements evolve. The challenge is managing these changes without breaking downstream consumers. Use **semantic versioning** for contracts:

- **MAJOR** (1.0 -> 2.0): Breaking change. Removing a column, changing a column type, tightening a constraint. Downstream consumers must update.
- **MINOR** (1.0 -> 1.1): Backward-compatible addition. Adding a new optional column. Downstream consumers can ignore the new column.
- **PATCH** (1.0.0 -> 1.0.1): Clarification or documentation update. No schema change.

In [12]:
# Schema evolution example

# v1.0: original schema
employee_schema_v1 = DataFrameSchema(
    columns={
        "employee_id": Column(str, nullable=False),
        "salary": Column(float, nullable=False, checks=[Check.in_range(20_000, 500_000)]),
        "department": Column(str, checks=[Check.isin(["Engineering", "Sales", "HR"])]),
    },
    name="employee_schema",
)

# v1.1: MINOR change -- add optional 'manager_id' column (backward compatible)
# Existing data without manager_id still passes v1.1 (nullable=True)
employee_schema_v1_1 = DataFrameSchema(
    columns={
        "employee_id": Column(str, nullable=False),
        "salary": Column(float, nullable=False, checks=[Check.in_range(20_000, 500_000)]),
        "department": Column(str, checks=[Check.isin(["Engineering", "Sales", "HR"])]),
        "manager_id": Column(str, nullable=True, required=False),  # new, optional
    },
    name="employee_schema",
)

# v2.0: MAJOR change -- rename 'salary' to 'annual_salary' (breaking)
employee_schema_v2 = DataFrameSchema(
    columns={
        "employee_id": Column(str, nullable=False),
        "annual_salary": Column(float, nullable=False, checks=[Check.in_range(20_000, 500_000)]),
        "department": Column(str, checks=[Check.isin(["Engineering", "Sales", "HR"])]),
        "manager_id": Column(str, nullable=True, required=False),
    },
    name="employee_schema",
)

# Show what happens when v1.0 data is validated against v2.0 schema
old_data = pd.DataFrame({
    "employee_id": ["EMP-00001"],
    "salary": [85000.0],       # old column name
    "department": ["Engineering"],
})

print("Validating v1.0 data against v1.0 schema:")
try:
    employee_schema_v1.validate(old_data)
    print("  PASS: v1.0 data is valid for v1.0 schema")
except pa.errors.SchemaError as e:
    print(f"  FAIL: {e}")

print("\nValidating v1.0 data against v2.0 schema (breaking change):")
try:
    employee_schema_v2.validate(old_data)
    print("  PASS (unexpected)")
except pa.errors.SchemaError as e:
    print(f"  FAIL (expected): column 'annual_salary' not found in DataFrame")

# Contract versioning best practice: use a VersionedContract wrapper
SCHEMA_REGISTRY = {
    "employee/v1.0": employee_schema_v1,
    "employee/v1.1": employee_schema_v1_1,
    "employee/v2.0": employee_schema_v2,
}

print("\nSchema registry:")
for version, schema in SCHEMA_REGISTRY.items():
    print(f"  {version}: {list(schema.columns.keys())}")

Validating v1.0 data against v1.0 schema:
  PASS: v1.0 data is valid for v1.0 schema

Validating v1.0 data against v2.0 schema (breaking change):
  FAIL (expected): column 'annual_salary' not found in DataFrame

Schema registry:
  employee/v1.0: ['employee_id', 'salary', 'department']
  employee/v1.1: ['employee_id', 'salary', 'department', 'manager_id']
  employee/v2.0: ['employee_id', 'annual_salary', 'department', 'manager_id']


## Section 6: Real-World Contract Example

A full ML training pipeline with 4 stages, each with input/output contracts. A schema violation at Stage 1 is caught immediately, preventing silent data corruption from reaching Stage 4 (model training).

In [13]:
# Define contracts for each pipeline stage

# Stage 1 output: raw ingested data
raw_schema = DataFrameSchema(
    columns={
        "employee_id": Column(str, nullable=False),
        "age":         Column(int, nullable=False, checks=[Check.in_range(18, 80)]),
        "salary":      Column(float, nullable=True),  # may have nulls before cleaning
        "department":  Column(str, nullable=False,
                              checks=[Check.isin(["Engineering", "Sales", "HR", "Finance", "Marketing"])]),
    },
    name="raw_employee_data_v1",
)

# Stage 2 output: cleaned data (nulls filled, outliers removed)
cleaned_schema = DataFrameSchema(
    columns={
        "employee_id": Column(str, nullable=False),
        "age":         Column(int, nullable=False, checks=[Check.in_range(18, 80)]),
        "salary":      Column(float, nullable=False, checks=[Check.in_range(20_000, 500_000)]),
        "department":  Column(str, nullable=False,
                              checks=[Check.isin(["Engineering", "Sales", "HR", "Finance", "Marketing"])]),
    },
    name="cleaned_employee_data_v1",
)

# Stage 3 output: engineered features
features_schema = DataFrameSchema(
    columns={
        "employee_id":       Column(str, nullable=False),
        "age_normalized":    Column(float, nullable=False, checks=[Check.in_range(0, 1)]),
        "salary_normalized": Column(float, nullable=False, checks=[Check.in_range(0, 1)]),
        "dept_Engineering":  Column(float, nullable=False, checks=[Check.isin([0.0, 1.0])]),
        "dept_Sales":        Column(float, nullable=False, checks=[Check.isin([0.0, 1.0])]),
        "dept_HR":           Column(float, nullable=False, checks=[Check.isin([0.0, 1.0])]),
    },
    name="engineered_features_v1",
    strict=False,  # allow additional dept_ columns
)

print("Pipeline contracts defined:")
for name, schema in [("Stage 1 (raw)", raw_schema), ("Stage 2 (cleaned)", cleaned_schema),
                     ("Stage 3 (features)", features_schema)]:
    print(f"  {name}: {list(schema.columns.keys())}")

Pipeline contracts defined:
  Stage 1 (raw): ['employee_id', 'age', 'salary', 'department']
  Stage 2 (cleaned): ['employee_id', 'age', 'salary', 'department']
  Stage 3 (features): ['employee_id', 'age_normalized', 'salary_normalized', 'dept_Engineering', 'dept_Sales', 'dept_HR']


In [14]:
# Pipeline functions with contracts enforced at each boundary

@pa.check_output(raw_schema)
def stage1_ingest(source: str) -> pd.DataFrame:
    """Stage 1: Ingest raw data. Contract checks output."""
    # In production: read from database, S3, Kafka, etc.
    return pd.DataFrame({
        "employee_id": ["EMP-00001", "EMP-00002", "EMP-00003", "EMP-00004"],
        "age":         [28, 45, 33, 52],
        "salary":      [85000.0, None, 95000.0, 110000.0],  # one null: OK at this stage
        "department":  ["Engineering", "Finance", "Sales", "Engineering"],
    })

@pa.check_input(raw_schema, obj_getter=0)
@pa.check_output(cleaned_schema)
def stage2_clean(raw: pd.DataFrame) -> pd.DataFrame:
    """Stage 2: Clean data. Contract checks both input and output."""
    df = raw.copy()
    median_salary = df["salary"].median()
    df["salary"] = df["salary"].fillna(median_salary)
    return df

@pa.check_input(cleaned_schema, obj_getter=0)
@pa.check_output(features_schema)
def stage3_engineer_features(cleaned: pd.DataFrame) -> pd.DataFrame:
    """Stage 3: Feature engineering. Contract checks both input and output."""
    df = cleaned.copy()
    df["age_normalized"] = (df["age"] - 18) / (80 - 18)
    salary_min, salary_max = 20_000, 500_000
    df["salary_normalized"] = (df["salary"] - salary_min) / (salary_max - salary_min)
    for dept in ["Engineering", "Sales", "HR", "Finance", "Marketing"]:
        df[f"dept_{dept}"] = (df["department"] == dept).astype(float)
    return df.drop(columns=["age", "salary", "department"])

def stage4_train_model(features: pd.DataFrame) -> str:
    """Stage 4: Train model. Receives validated features."""
    # In production: fit sklearn/XGBoost/etc.
    return f"model trained on {len(features)} rows, {len(features.columns)} features"

# Run the full pipeline
print("Running ML pipeline with contract enforcement:")
print()

raw_data   = stage1_ingest("production_db");          print(f"Stage 1 (ingest): {len(raw_data)} rows [PASSED]")
clean_data = stage2_clean(raw_data);                   print(f"Stage 2 (clean):  {len(clean_data)} rows [PASSED]")
features   = stage3_engineer_features(clean_data);     print(f"Stage 3 (features): {len(features)} rows, {len(features.columns)} cols [PASSED]")
model_info = stage4_train_model(features);             print(f"Stage 4 (train):  {model_info} [PASSED]")
print()
print("All pipeline stages completed. Contracts enforced at every boundary.")

Running ML pipeline with contract enforcement:

Stage 1 (ingest): 4 rows [PASSED]
Stage 2 (clean):  4 rows [PASSED]
Stage 3 (features): 4 rows, 8 cols [PASSED]
Stage 4 (train):  model trained on 4 rows, 8 features [PASSED]

All pipeline stages completed. Contracts enforced at every boundary.


In [15]:
# Demonstrate contract catching a violation early
# Simulate what happens when Stage 1 receives corrupted data with a schema change

print("Simulating upstream schema change: 'department' column renamed to 'dept'")
print()

def stage1_ingest_broken(source: str) -> pd.DataFrame:
    """Broken ingestion: upstream renamed 'department' to 'dept'."""
    return pd.DataFrame({
        "employee_id": ["EMP-00001", "EMP-00002"],
        "age":         [28, 45],
        "salary":      [85000.0, 95000.0],
        "dept":        ["Engineering", "Finance"],  # wrong column name
    })

try:
    raw_data = raw_schema.validate(stage1_ingest_broken("production_db"))
    clean_data = stage2_clean(raw_data)
    features = stage3_engineer_features(clean_data)
    model_info = stage4_train_model(features)
    print("Pipeline completed (contract did NOT catch the error -- this is bad!)")
except pa.errors.SchemaError as e:
    print(f"[CONTRACT CAUGHT IT] Stage 1 output validation failed:")
    print(f"  Error: column 'department' not found in DataFrame")
    print()
    print("Without contracts, this error would propagate silently to Stage 4 (model training).")
    print("With contracts, it is caught at Stage 1 boundary immediately.")

Simulating upstream schema change: 'department' column renamed to 'dept'

[CONTRACT CAUGHT IT] Stage 1 output validation failed:
  Error: column 'department' not found in DataFrame

Without contracts, this error would propagate silently to Stage 4 (model training).
With contracts, it is caught at Stage 1 boundary immediately.


## Key Takeaways

1. **Data contracts make implicit assumptions explicit.** Every data pipeline has assumptions about column names, types, value ranges. Contracts encode those assumptions as machine-enforceable specifications.

2. **Pandera validates DataFrames at pipeline boundaries.** Use `@pa.check_output` on producers and `@pa.check_input` on consumers. When data fails, Pandera raises a `SchemaError` with a precise description.

3. **Statistical contracts catch distribution shifts.** Schema checks catch structural problems. Statistical checks catch distributional problems (mean shifted, variance increased, category proportions changed).

4. **Pydantic validates individual records.** For streaming pipelines, validate each event before processing. Quarantine invalid records rather than dropping them silently or letting them corrupt downstream state.

5. **Contract tests belong in CI/CD.** A broken contract caught in CI is far cheaper than one caught in production after a model is already trained on corrupted features.

6. **Version contracts like code.** Distinguish breaking changes (MAJOR) from backward-compatible additions (MINOR). Pin contract versions in pipeline configuration.

**The data contract mindset**: every interface between a data producer and a data consumer is a contract. Make it explicit, make it testable, and fail loudly when it is violated.